In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
from datasets import load_dataset

from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

In [ ]:
dataset = load_dataset('glue', 'sst2')

In [ ]:
max_length = 128

def tokenize_fn(example):
    enc = tokenizer(
        example['sentence'],
        padding=True,
        truncation=True,
        max_length=max_length
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': example['label']
    }

dataset = dataset.map(tokenize_fn, batched=True, batch_size=2000)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
val_data = dataset['validation']


In [ ]:
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc_result = accuracy.compute(predictions=predictions, references=labels)
    f1_result = f1.compute(predictions=predictions, references=labels, average='weighted')
    return {**acc_result, **f1_result}


In [ ]:
training_args = TrainingArguments(
    output_dir='./bert-sst2',
    per_device_eval_batch_size=8,
    report_to='none',
    do_train=False,
    do_eval=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=val_data,
    compute_metrics=compute_metrics
)


In [ ]:

final_eval = trainer.evaluate()
print('Final Evaluation : \n')
print(f"Eval Loss: {final_eval['eval_loss']:.4f} | Eval Accuracy: {final_eval['eval_accuracy']:.4f} | Eval F1 Score: {final_eval['eval_f1']:.4f}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Final Evaluation : 

Eval Loss: 0.7470 | Eval Accuracy: 0.4908 | Eval F1 Score: 0.3232
